# Momentum Wheels

## Introduction

A spacecraft stores the angular impulse of the environment's torques in its
momentum (reaction) wheels. Each wheel spins about one axis fixed in the body and
holds only so much momentum along it, so how much the spacecraft can store depends
on the wheels it has, how they are mounted and which way the momentum points.

COASTSim describes the wheels under
`spacecraft_bus.attitude_control.stored_momentum.wheels`:

* Each `MomentumWheel` has a `spin_axis`, a `capacity_n_m_s`, and a position and
  size for drawing.
* `WheelArray` shares the momentum the wheels hold among them and gives the set's
  total capacity: what it holds whichever way the momentum points.
* A run shares the stored momentum among the wheels at every step, reports each
  wheel's share in the telemetry, and calls for a thruster dump when any one wheel
  reaches the planning level of its own capacity.
* `plot_spacecraft_3d` draws the wheels inside the bus.

This notebook builds two wheel sets, looks at how their capacity varies with
direction, draws them on the spacecraft, and runs a day with each wheel limited by
its own capacity. It follows on from `Momentum_Tracking_Example.ipynb`, which
covers the torques themselves.

## Step 1: Import Libraries

In [ ]:
%matplotlib inline
from datetime import datetime, timedelta, timezone

import matplotlib.pyplot as plt
import numpy as np
from rust_ephem import EarthLimbConstraint, SunConstraint, TLEEphemeris

from conops import create_ditl
from conops.common import ACSMode
from conops.config import (
    AttitudeControlSystem,
    Battery,
    Constraint,
    GroundStationRegistry,
    MissionConfig,
    MomentumWheel,
    PanelGeometry,
    Payload,
    RadiatorConfiguration,
    SolarPanel,
    SolarPanelSet,
    SpacecraftBus,
    StarTrackerConfiguration,
    StoredMomentumConfig,
    Telescope,
    TelescopeConfig,
    TelescopeType,
    ThrusterDumpConfig,
    WheelArray,
)
from conops.targets import Pointing
from conops.visualization import plot_spacecraft_3d

## Step 2: Two Sets of Wheels

Three wheels along the body axes is the least that can hold momentum in every
direction. Four wheels in a pyramid about the boresight (body +X) is a common
arrangement with a spare. Every wheel here holds 4 N m s. The spin axes are
normalised for you.

In [ ]:
WHEEL_CAPACITY = 4.0  # N m s

orthogonal = [
    MomentumWheel(name="RW-X", spin_axis=(1, 0, 0), capacity_n_m_s=WHEEL_CAPACITY),
    MomentumWheel(name="RW-Y", spin_axis=(0, 1, 0), capacity_n_m_s=WHEEL_CAPACITY),
    MomentumWheel(name="RW-Z", spin_axis=(0, 0, 1), capacity_n_m_s=WHEEL_CAPACITY),
]
pyramid = [
    MomentumWheel(name=f"RW{i + 1}", spin_axis=axis, capacity_n_m_s=WHEEL_CAPACITY)
    for i, axis in enumerate([(1, 1, 1), (1, -1, 1), (1, -1, -1), (1, 1, -1)])
]

for label, wheels in [
    ("Three orthogonal", orthogonal),
    ("Four-wheel pyramid", pyramid),
]:
    array = WheelArray(wheels)
    print(f"{label}: total capacity {array.capacity_n_m_s:.2f} N m s")
    for wheel in wheels:
        axis = ", ".join(f"{x:5.2f}" for x in wheel.spin_axis)
        print(
            f"  {wheel.name:<5} axis ({axis})  capacity {wheel.capacity_n_m_s:g} N m s"
        )

## Step 3: Sharing Momentum Among the Wheels

`WheelArray.wheel_momenta_n_m_s` shares a body-frame momentum among the wheels.
Three wheels have only one way to make it up. Four have many, and the one taken
has the least sum of squared fractions of each wheel's capacity, so a larger wheel
takes more. `utilisation` is the fullest wheel's fraction of its capacity: the set
is at its limit when that reaches 1.

In [ ]:
momentum = np.array([2.0, -1.0, 3.0])  # N m s, body frame
print(f"Body momentum {momentum}, norm {np.linalg.norm(momentum):.2f} N m s\n")
for label, wheels in [
    ("Three orthogonal", orthogonal),
    ("Four-wheel pyramid", pyramid),
]:
    array = WheelArray(wheels)
    shares = array.wheel_momenta_n_m_s(momentum)
    print(label)
    for wheel, share in zip(wheels, shares):
        print(
            f"  {wheel.name:<5} {share:6.2f} N m s  ({abs(share) / wheel.capacity_n_m_s:4.0%} of capacity)"
        )
    rebuilt = shares @ np.array([w.spin_axis for w in wheels])
    print(f"  shares along the spin axes add back to {np.round(rebuilt, 6)}")
    print(f"  fullest wheel at {array.utilisation(momentum):.0%}\n")

## Step 4: The Capacity Depends on Direction

A single wheel fills at 4 N m s. Momentum pointing between the spin axes is shared,
so the set holds more of it before any one wheel fills. The plot shows, for
momentum in the body X-Y and Y-Z planes, how much each set holds along every
direction. The dashed circle is the total capacity: the least over all directions,
which is the one figure the planning level and the capacity are quoted against.

In [ ]:
angles = np.linspace(0, 2 * np.pi, 721)
planes = {
    "X-Y plane": np.stack(
        [np.cos(angles), np.sin(angles), np.zeros_like(angles)], axis=1
    ),
    "Y-Z plane": np.stack(
        [np.zeros_like(angles), np.cos(angles), np.sin(angles)], axis=1
    ),
}
fig, axes = plt.subplots(1, 2, figsize=(10, 4.8), subplot_kw={"projection": "polar"})
for ax, (plane, directions) in zip(axes, planes.items()):
    for label, wheels, color in [
        ("Three orthogonal", orthogonal, "tab:blue"),
        ("Four-wheel pyramid", pyramid, "tab:orange"),
    ]:
        array = WheelArray(wheels)
        reach = np.array([1.0 / array.utilisation(d) for d in directions])
        ax.plot(angles, reach, color=color, label=label)
        ax.plot(
            angles,
            np.full_like(angles, array.capacity_n_m_s),
            color=color,
            ls="--",
            lw=0.8,
        )
    ax.set_title(f"Momentum held, {plane} (N m s)")
    ax.set_rmax(9.5)
axes[0].legend(fontsize=8, loc="lower left", bbox_to_anchor=(-0.25, -0.15))
fig.tight_layout()
plt.show()

for label, wheels in [
    ("Three orthogonal", orthogonal),
    ("Four-wheel pyramid", pyramid),
]:
    array = WheelArray(wheels)
    rng = np.random.default_rng(0)
    directions = rng.normal(size=(20000, 3))
    directions /= np.linalg.norm(directions, axis=1, keepdims=True)
    reach = 1.0 / np.abs(directions @ array.fractions.T).max(axis=1)
    print(
        f"{label}: holds between {reach.min():.2f} and {reach.max():.2f} N m s, by direction"
    )

## Step 5: The Wheels on the Spacecraft

The wheels go into the stored-momentum configuration. With no `capacity_n_m_s`
given, the wheels set the limits. `total_capacity_n_m_s` is the figure the
planning level is a fraction of.

The spacecraft is the one from the momentum tracking example: an inertia tensor of
1000, 1200 and 800 kg m², a 0.5 m imager along the boresight and one array on the
+Y side, in a 600 km orbit sampled every 10 seconds.

In [ ]:
begin = datetime(2025, 12, 1, tzinfo=timezone.utc)
end = begin + timedelta(hours=24)
ephemeris = TLEEphemeris(begin=begin, end=end, step_size=10, tle="example.tle")

INERTIA = [[1000.0, 0.0, 0.0], [0.0, 1200.0, 0.0], [0.0, 0.0, 800.0]]


def make_config(**momentum) -> MissionConfig:
    config = MissionConfig(
        ground_stations=GroundStationRegistry(stations=[]),
        constraint=Constraint(
            sun_constraint=SunConstraint(min_angle=45),
            earth_constraint=EarthLimbConstraint(min_angle=10),
        ),
        solar_panel=SolarPanelSet(
            panels=[
                SolarPanel(
                    name="Array",
                    normal=(0.0, 1.0, 0.0),
                    max_power=600.0,
                    geometry=PanelGeometry(
                        center_m=(0.0, 1.8, 0.0),
                        u=(1.0, 0.0, 0.0),
                        v=(0.0, 0.0, 1.0),
                        width_m=3.0,
                        height_m=1.2,
                    ),
                )
            ]
        ),
        battery=Battery(watthour=100_000.0),
        spacecraft_bus=SpacecraftBus(
            inertia_tensor_body_kg_m2=INERTIA,
            attitude_control=AttitudeControlSystem(
                max_slew_rate=0.5,
                slew_acceleration=0.25,
                settle_time=30.0,
                stored_momentum=StoredMomentumConfig(
                    gravity_gradient_enabled=True,
                    solar_radiation_pressure_enabled=True,
                    planning_level=0.8,
                    thruster_dump=ThrusterDumpConfig(duration_seconds=120.0),
                    **momentum,
                ),
            ),
            star_trackers=StarTrackerConfiguration(
                star_trackers=[], min_functional_trackers=0, modes_require_lock=[]
            ),
            radiators=RadiatorConfiguration(radiators=[]),
        ),
        payload=Payload(
            instruments=[
                Telescope(
                    name="Imager",
                    optics=TelescopeConfig(
                        aperture_m=0.5,
                        focal_length_m=3.5,
                        telescope_type=TelescopeType.RITCHEY_CHRETIEN,
                        tube_length_m=1.2,
                    ),
                )
            ]
        ),
    )
    config.constraint.ephem = ephemeris
    return config


config = make_config(wheels=pyramid)
settings = config.spacecraft_bus.attitude_control.stored_momentum
print(
    f"{len(settings.wheels)} wheels, total capacity {settings.total_capacity_n_m_s:.2f} N m s"
)
print(
    f"A dump is called for at {settings.dump_level_n_m_s:.2f} N m s, or any wheel at {settings.planning_level:.0%} of its own"
)

`plot_spacecraft_3d` draws each wheel as a disc turned to its spin axis, labelled
with its capacity, with an arrow along the axis. With no `position_m` the wheels
are placed inside the bus along their spin axes, and the bus is drawn translucent
so they show. `show_wheels=False` hides them.

In [ ]:
fig = plot_spacecraft_3d(config, title="Four-wheel pyramid")
fig.show()

A wheel's `position_m`, `radius_m` and `thickness_m` place and size it. They are
for the drawing only: where a wheel sits does not change the momentum it holds.

In [ ]:
placed = [
    wheel.model_copy(update={"position_m": position, "radius_m": 0.12})
    for wheel, position in zip(
        orthogonal, [(-0.6, 0.0, 0.0), (-0.6, 0.3, 0.0), (-0.6, 0.0, 0.3)]
    )
]
fig = plot_spacecraft_3d(
    make_config(wheels=placed), title="Three wheels at the back of the bus"
)
fig.show()

## Step 6: A Day Within the Wheels

One target held all day: an inertial hold makes the gravity-gradient torque
secular, so the wheels fill steadily. The telemetry's `wheel_momentum_n_m_s` is
each wheel's share at every step. Dispatch calls for a thruster dump when the
fullest wheel reaches the planning level of its capacity, and the dump empties
them all.

The wheels do not fill alike. Each takes the part of the momentum that lies along
its own spin axis, so at this attitude RW2 fills fastest and sets when the dumps
come, while RW3 is hardly used.

The plot leaves out the few samples taken while slewing. There the wheels also
give up and take back the body's own momentum, which this configuration does not
count against them (`slew_momentum_enabled` is off); with it on, slews are
admitted against each wheel too.

In [ ]:
def run(config: MissionConfig):
    target = Pointing(
        config=config,
        ra=0.0,
        dec=0.0,
        obsid=900,
        name="Held",
        merit=100.0,
        ss_min=600,
        ss_max=24 * 3600,
    )
    target.exptime = 24 * 3600
    ditl = create_ditl(config, [target], begin, end)
    ditl.step_size = 10
    assert ditl.calc()
    return ditl


def series(ditl, name):
    records = ditl.telemetry.housekeeping
    hours = (
        np.array([hk.timestamp.timestamp() for hk in records]) - begin.timestamp()
    ) / 3600
    values = list(getattr(records, name))
    keep = [v is not None for v in values]
    return hours[keep], np.array([v for v in values if v is not None], dtype=float)


def dump_hours(ditl):
    return [
        (float(e.begin) - begin.timestamp()) / 3600 for e in ditl.plan if e.desaturation
    ]


def holding(ditl):
    """The samples taken while not slewing."""
    modes = np.array(ditl.mode)[: len(ditl.telemetry.housekeeping)]
    return modes != ACSMode.SLEWING


by_wheel = run(config)
hours, shares = series(by_wheel, "wheel_momentum_n_m_s")
shares = np.where(holding(by_wheel)[:, None], shares, np.nan)

fig, ax = plt.subplots(figsize=(10, 4))
for index, wheel in enumerate(pyramid):
    ax.plot(hours, shares[:, index], lw=1, label=wheel.name)
for sign in (1, -1):
    ax.axhline(
        sign * WHEEL_CAPACITY,
        color="tab:red",
        ls="--",
        lw=1,
        label="wheel capacity" if sign == 1 else None,
    )
    ax.axhline(
        sign * settings.planning_level * WHEEL_CAPACITY,
        color="tab:orange",
        ls=":",
        lw=1,
        label="planning level" if sign == 1 else None,
    )
for dump in dump_hours(by_wheel):
    ax.axvline(dump, color="0.6", lw=0.8)
ax.set_xlim(0, 24)
ax.set_xlabel("Hours")
ax.set_ylabel("Wheel momentum (N m s)")
ax.set_title("Each wheel's share of the stored momentum; grey lines are dumps")
ax.legend(fontsize=8, ncol=3, loc="upper left")
fig.tight_layout()
plt.show()

print("Dumps at " + ", ".join(f"{h:.1f} h" for h in dump_hours(by_wheel)))
print(
    f"Fullest any wheel got: {np.nanmax(np.abs(shares)):.2f} N m s of {WHEEL_CAPACITY:g}, "
    f"against a planning level of {settings.planning_level * WHEEL_CAPACITY:.2f}"
)

## Step 7: Against a Single Figure

Before wheels could be described, the capacity was one number, the same about
every axis. Giving `capacity_n_m_s` still does that, and given alongside `wheels`
it replaces their limits. Here the same day runs with the pyramid's total capacity
as a single figure. The momentum's norm reaches that figure's planning level
before any wheel reaches its own, unless the momentum happens to point along the
wheels' weakest direction, so the single figure dumps sooner.

The second plot shows what that costs. Each wheel's share is drawn against its
own capacity and planning level, solid where the wheels set the limits and dashed
where the single figure does: under the single figure a dump comes while every
wheel still has room.

In [ ]:
single = run(make_config(wheels=pyramid, capacity_n_m_s=settings.total_capacity_n_m_s))
runs = [
    (by_wheel, "limited by each wheel", "-"),
    (single, "limited by one figure", "--"),
]

fig, (ax, wheel_ax) = plt.subplots(
    2, 1, figsize=(10, 7.6), sharex=True, height_ratios=[1, 1.2]
)
for (ditl, label, _), color in zip(runs, ["tab:orange", "tab:blue"]):
    h, norm = series(ditl, "stored_momentum_norm_n_m_s")
    ax.plot(h, norm, color=color, lw=1, label=label)
ax.axhline(
    settings.total_capacity_n_m_s,
    color="tab:red",
    ls="--",
    lw=1,
    label="total capacity",
)
ax.axhline(
    settings.dump_level_n_m_s,
    color="tab:orange",
    ls=":",
    lw=1,
    label="planning level of the total",
)
ax.set_ylabel("Stored momentum norm (N m s)")
ax.set_title("The same day, limited by each wheel and by one figure")
ax.legend(fontsize=8, loc="upper left")

fullest = {}
for ditl, label, style in runs:
    h, wheel_shares = series(ditl, "wheel_momentum_n_m_s")
    wheel_shares = np.where(holding(ditl)[:, None], wheel_shares, np.nan)
    fullest[label] = np.nanmax(np.abs(wheel_shares))
    for index, wheel in enumerate(pyramid):
        wheel_ax.plot(
            h,
            wheel_shares[:, index],
            color=f"C{index}",
            ls=style,
            lw=1,
            label=f"{wheel.name}, {label}",
        )
for sign in (1, -1):
    wheel_ax.axhline(
        sign * WHEEL_CAPACITY,
        color="tab:red",
        ls="--",
        lw=1,
        label="wheel capacity" if sign == 1 else None,
    )
    wheel_ax.axhline(
        sign * settings.planning_level * WHEEL_CAPACITY,
        color="tab:orange",
        ls=":",
        lw=1,
        label="planning level of a wheel" if sign == 1 else None,
    )
wheel_ax.set_xlim(0, 24)
wheel_ax.set_xlabel("Hours")
wheel_ax.set_ylabel("Wheel momentum (N m s)")
wheel_ax.set_title("Each wheel's share on the same day, against its own limits")
wheel_ax.legend(fontsize=7, ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.16))
fig.tight_layout()
plt.show()

_, norm = series(by_wheel, "stored_momentum_norm_n_m_s")
for ditl, label in [(by_wheel, "Each wheel"), (single, "One figure")]:
    print(f"{label:<11} dumps at " + ", ".join(f"{h:.1f} h" for h in dump_hours(ditl)))
print(
    f"Limited by each wheel, the norm reaches {norm.max():.2f} N m s; "
    f"the single figure dumps at {settings.dump_level_n_m_s:.2f} N m s"
)
for label, value in fullest.items():
    print(f"Fullest any wheel got, {label}: {value:.2f} N m s of {WHEEL_CAPACITY:g}")

## Step 8: Wheels of Different Sizes

The wheels need not match. Swapping the pyramid's first wheel for one half the
size lowers the total capacity, and the sharing gives the small wheel less to
hold. The day's dumps then come when whichever wheel is fullest, as a fraction of
its own capacity, reaches the planning level.

In [ ]:
mixed = [
    pyramid[0].model_copy(update={"capacity_n_m_s": 2.0, "name": "RW1 (small)"}),
    *pyramid[1:],
]
mixed_array = WheelArray(mixed)
print(
    f"Total capacity {mixed_array.capacity_n_m_s:.2f} N m s, from {WheelArray(pyramid).capacity_n_m_s:.2f} with four equal wheels"
)
for wheel, share in zip(mixed, mixed_array.wheel_momenta_n_m_s(momentum)):
    print(
        f"  {wheel.name:<12} {share:6.2f} N m s  ({abs(share) / wheel.capacity_n_m_s:4.0%} of capacity)"
    )

mixed_run = run(make_config(wheels=mixed))
print()
for ditl, label in [(mixed_run, "One small wheel"), (by_wheel, "Four equal wheels")]:
    print(f"{label:<18} dumps at " + ", ".join(f"{h:.1f} h" for h in dump_hours(ditl)))

## Summary

* `MomentumWheel` gives a wheel its spin axis, its capacity and, for drawing, its
  position and size; a list of them goes in `stored_momentum.wheels`.
* `WheelArray` shares a body momentum among the wheels and gives the total
  capacity: what the set holds along its weakest direction.
* The set holds more along directions several wheels share, so the limit depends
  on where the stored momentum points in the body, and changes as the spacecraft
  turns.
* A run reports each wheel's share as `wheel_momentum_n_m_s`, calls for a dump when
  a wheel reaches the planning level of its own capacity, and with
  `enforce_capacity` certifies every slew and hold against each wheel.
* `capacity_n_m_s` alongside the wheels replaces their limits with one figure.
* More than three wheels share by one fixed law; redistributing momentum among
  them, wheel speeds and failed wheels are not modelled.